# Calculation Demonstration: Step-by-Step Overshoot Walkthrough

This notebook walks through **every input variable and formula** used by `scripts/calculate_overshoot.py`,
using **USA (area_code = 231)** and **year 2023** as a worked example.

> *"The Ecological Footprint measures how much biologically productive land and water area a population
> requires to produce the resources it consumes and to absorb the waste it generates, using prevailing
> technology."*
> — Borucke et al. (2013), *Ecological Indicators* 24: 518–533

---

### Conceptual Framework

The **National Footprint Accounts** (NFA) methodology, developed by the Global Footprint Network,
translates resource flows into a single metric: **global hectares (gha)**. A global hectare represents
one hectare of biologically productive space with *world-average* productivity across all land types
(Galli et al. 2007). This allows direct comparison of demand (Ecological Footprint) against supply
(Biocapacity).

The **production-based** Footprint used in this calculator counts resources where they are produced:

$$\text{EF}_{\text{production}} = \sum_i \frac{P_i}{Y_{W,i}} \times \text{EQF}_i$$

where $P_i$ is the amount of product $i$ harvested and $Y_{W,i}$ is the world average yield
(Borucke et al. 2013, Eq. 1). The official NFA additionally adjusts for trade
($\text{EF}_\text{consumption} = \text{EF}_\text{production} + \text{EF}_\text{imports} - \text{EF}_\text{exports}$),
which this implementation does not.

Biocapacity is computed as:

$$\text{BC} = \sum_i A_{N,i} \times \text{YF}_{N,i} \times \text{EQF}_i$$

where $A_{N,i}$ is national area for land type $i$, $\text{YF}$ is the yield factor, and $\text{EQF}$ is the equivalence factor (Borucke et al. 2013, Eq. 5).

---

### Component Overview

**6 EF Components** (demand side):

| # | Component | What it measures | EQF | Key data source |
|---|-----------|-----------------|-----|-----------------|
| 1 | **Cropland** | Area to grow crops at world-average yields | 2.1 | FAOSTAT Production (QCL) |
| 2 | **Grazing** | Pasture area to feed livestock | 0.5 | FAOSTAT Livestock + GLEAM |
| 3 | **Forest** | Forest area to sustain roundwood harvest | 1.3 | FAOSTAT Forestry (FO) |
| 4 | **Fishing** | Marine area to sustain fish catch (PPR method) | 0.4 | FAOSTAT Food Balance Sheets |
| 5 | **Built-up** | Land covered by infrastructure | 2.2 | FAOSTAT Land Use (RL) |
| 6 | **Carbon** | Forest area to sequester CO₂ emissions | 1.3 | FAOSTAT Emissions (GT) |

**5 BC Components** (supply side): same land types except carbon — forest BC already represents the sequestration capacity that the carbon EF draws upon (Borucke et al. 2013, §3.5).

---

### Glossary of Key Terms

| Abbreviation | Term | Definition |
|---|---|---|
| **EF** | Ecological Footprint | Total bioproductive area required to produce resources consumed and absorb CO₂ waste (gha) |
| **BC** | Biocapacity | Total bioproductive area available within a country to regenerate resources (gha) |
| **gha** | Global Hectare | A hectare normalized to world-average bioproductivity across all land types (Galli et al. 2007) |
| **EQF** | Equivalence Factor | Converts physical hectares of a specific land type into gha; derived from GAEZ suitability indices (Borucke et al. 2013) |
| **YF** | Yield Factor | Ratio of national yield to world yield for a land type; captures how productive a country's land is relative to the global mean |
| **PPR** | Primary Production Required | Marine phytoplankton production needed to sustain a fish catch, via the trophic-level method (Pauly & Christensen 1995) |
| **TL** | Trophic Level | Position in the marine food chain (1 = phytoplankton, 2 = herbivores, 3+ = predators) |
| **TE** | Transfer Efficiency | Fraction of energy transferred per trophic level; ~10% (Pauly & Christensen 1995) |
| **AFCS** | Annual Forest Carbon Sequestration | Average rate at which world forests absorb carbon: 0.73 t C/ha/yr (Mancini et al. 2016) |
| **NAI** | Net Annual Increment | World-average forest biomass growth: 1.81 m³/ha/yr (UNECE/FAO TBFRA 2000) |
| **NPP** | Net Primary Productivity | Annual plant biomass production; for grassland: 2.45 t DM/ha/yr (Monfreda et al. 2008) |
| **DM** | Dry Matter | Biomass weight excluding water; the standard unit for feed demand and grassland productivity |
| **FBS** | Food Balance Sheet | FAOSTAT dataset tracking production, imports, exports, and use by commodity |

### Key References

| # | Reference | What it provides |
|---|-----------|-----------------|
| 1 | Wackernagel et al. (2002), *PNAS* 99(14): 9266–9271 | Original EF framework & EQF values |
| 2 | Borucke et al. (2013), *Ecol. Indicators* 24: 518–533 | Definitive NFA methodology paper |
| 3 | Galli et al. (2007), *Int. J. Ecodynamics* 2(4): 250–257 | Mathematical exposition of EF/BC framework |
| 4 | Monfreda et al. (2004), *Land Use Policy* 21: 231–246 | National natural capital accounts methodology |
| 5 | Lin et al. (2018), *Resources* 7(3): 58 | NFA 2012–2018 updates including AFCS revision |
| 6 | Mancini et al. (2016), *Ecol. Indicators* 61: 390–403 | Refined carbon Footprint calculation |
| 7 | Pauly & Christensen (1995), *Nature* 374: 255–257 | PPR method & trophic parameters |
| 8 | Khatiwala et al. (2009), *Nature* 462: 346–349 | Ocean CO₂ uptake history |
| 9 | Monfreda et al. (2008), *Global Biogeochem. Cycles* 22(1) | Grassland NPP values |
| 10 | Gulland (1971), *The Fish Resources of the Ocean* | Global sustainable catch estimate |

**Demo Country:** United States of America (area_code = 231)
**Demo Year:** 2023

---
## Section 1: Load Data & Reference Parameters

This section loads all input datasets and displays the reference parameter CSVs that define the
constants used in every calculation.

**Data pipeline:** `scripts/fetch_footprint_data.py` downloads bulk FAOSTAT ZIP archives and
normalizes them into clean CSVs. The calculator then reads these CSVs and applies the formulas
documented in the NFA methodology (Borucke et al. 2013).

In [1]:
import os
import zipfile
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 25)
pd.set_option("display.max_rows", 60)

# ── Paths ───────────────────────────────────────────────────────────────────
ROOT = Path(".").resolve().parent
DATA = ROOT / "data"

DEMO_COUNTRY = 231       # USA
DEMO_YEAR = 2023
YEAR_MIN, YEAR_MAX = 2014, 2023

def load(rel_path, year_filter=True):
    """Load a FAOSTAT CSV, optionally filter to 2014-2023."""
    df = pd.read_csv(DATA / rel_path)
    if year_filter and "Year" in df.columns:
        df = df[df["Year"].between(YEAR_MIN, YEAR_MAX)]
    return df

# ── 8 input data files ───────────────────────────────────────────────────────
crop_prod    = load("01_cropland/crop_production.csv")
cropland_area = load("01_cropland/cropland_area.csv")
livestock    = load("02_grazing/livestock_stocks.csv")
pasture_area = load("02_grazing/pasture_area.csv")
timber       = load("03_forest/timber_production.csv")
forest_area  = load("03_forest/forest_area.csv")
fish_fbs     = load("04_fishing/fish_supply_fbs.csv")
co2_energy   = load("06_carbon/co2_energy_faostat.csv")
population   = load("07_population/population.csv")

# Land-use ZIP
land_use_zip = DATA / "raw" / "Inputs_LandUse_E_All_Data_(Normalized).zip"
with zipfile.ZipFile(land_use_zip) as z:
    csv_name = [n for n in z.namelist() if n.endswith(".csv") and "Data" in n][0]
    land_use_all = pd.read_csv(z.open(csv_name), encoding="latin-1")
land_use_all = land_use_all[land_use_all["Year"].between(YEAR_MIN, YEAR_MAX)]

# Results for validation
results = pd.read_csv(DATA / "overshoot_results.csv")
global_summary = pd.read_csv(DATA / "overshoot_global_summary.csv")

# Helper: get stored result for a country+year
def stored(area_code, year, col):
    row = results[(results["area_code"] == area_code) & (results["year"] == year)]
    return row[col].iloc[0] if len(row) > 0 else None

print("All data loaded.")

All data loaded.


### Reference Parameter CSVs

These CSV files contain the **static parameters** sourced from the academic literature. Each file
includes the parameter value, units, and original publication reference.

> *"NFAs are specifically constructed to yield conservative estimates of global overshoot. On the supply
> side, biocapacity is overestimated... On the demand side, Ecological Footprint is underestimated as it
> does not track freshwater consumption, soil erosion, or GHG emissions other than CO₂."*
> — Borucke et al. (2013)

**Legend for the tables below:**

| CSV File | Contains | Used in |
|---|---|---|
| `equivalence_factors_reference_1999.csv` | EQF values (gha/ha) for each land type | All EF and BC calculations |
| `gleam_feed_coefficients.csv` | Pasture dry matter demand per livestock head | Grazing EF (Section 5) |
| `ppr_calculation_parameters.csv` | PPR method constants (TE, DR, WC) | Fishing EF (Section 9) |
| `carbon_sequestration_parameters.csv` | AFCS rate and ocean CO₂ absorption fraction | Carbon EF (Section 12) |
| `forest_nai_reference.csv` | World-average forest growth rate | Forest EF (Section 7) |

In [2]:
print("=" * 70)
print("EQUIVALENCE FACTORS (gha per ha)")
print("=" * 70)
eqf_df = pd.read_csv(DATA / "09_factors" / "equivalence_factors_reference_1999.csv")
display(eqf_df)

print("\n" + "=" * 70)
print("GLEAM FEED COEFFICIENTS (pasture DM demand per head)")
print("=" * 70)
gleam_df = pd.read_csv(DATA / "02_grazing" / "gleam_feed_coefficients.csv")
display(gleam_df)

print("\n" + "=" * 70)
print("PPR CALCULATION PARAMETERS")
print("=" * 70)
ppr_df = pd.read_csv(DATA / "04_fishing" / "ppr_calculation_parameters.csv")
display(ppr_df)

print("\n" + "=" * 70)
print("CARBON SEQUESTRATION PARAMETERS")
print("=" * 70)
carbon_params_df = pd.read_csv(DATA / "10_reference_parameters" / "carbon_sequestration_parameters.csv")
display(carbon_params_df)

print("\n" + "=" * 70)
print("FOREST NAI REFERENCE")
print("=" * 70)
nai_df = pd.read_csv(DATA / "10_reference_parameters" / "forest_nai_reference.csv")
display(nai_df)

EQUIVALENCE FACTORS (gha per ha)


,land_type,equivalence_factor_gha_per_ha,notes
0,Cropland,2.1000,Wackernagel et al. 2002 (Table 1)
1,Built-up land,2.2000,Assumed same productivity as cropland it replaces
2,Forest,1.3000,Wackernagel et al. 2002 (Table 1)
3,Grazing land,0.5000,Wackernagel et al. 2002 (Table 1)
4,Fishing grounds,0.4000,Wackernagel et al. 2002 (Table 1)
5,Carbon (fossil fuel),1.3000,Uses forest equivalence factor (sequestration ...



GLEAM FEED COEFFICIENTS (pasture DM demand per head)


,animal_type,total_dmi_kg_per_day,pasture_fraction,crop_fraction,residue_fraction,pasture_dmi_kg_per_day,annual_pasture_demand_t_dm,source
0,Cattle (dairy),14.5000,0.4500,0.3000,0.2500,6.5300,2.3820,FAO GLEAM 3.0 (global averages)
1,Cattle (beef),9.8000,0.6500,0.1500,0.2000,6.3700,2.3250,FAO GLEAM 3.0 (global averages)
2,Buffalo,12.0000,0.5500,0.2000,0.2500,6.6000,2.4090,FAO GLEAM 3.0 (global averages)
3,Sheep,1.8000,0.7000,0.1000,0.2000,1.2600,0.4600,FAO GLEAM 3.0 (global averages)
4,Goats,1.5000,0.7200,0.0800,0.2000,1.0800,0.3940,FAO GLEAM 3.0 (global averages)
5,Pigs,2.5000,0.0000,0.7500,0.2500,0.0000,0.0000,FAO GLEAM 3.0 (global averages)
6,Chickens (layers),0.1200,0.0000,0.9000,0.1000,0.0000,0.0000,FAO GLEAM 3.0 (global averages)
7,Chickens (broilers),0.1000,0.0000,0.9000,0.1000,0.0000,0.0000,FAO GLEAM 3.0 (global averages)



PPR CALCULATION PARAMETERS


,parameter,value,unit,source,notes
0,transfer_efficiency,0.1013,fraction,Pauly & Christensen 1995,Efficiency between trophic levels (10.13%)
1,discard_rate,1.2700,ratio,Pauly & Christensen 1995,1.27 = 0.27 t bycatch per tonne harvested
2,wet_weight_to_carbon,9.0000,ratio,Sea Around Us / Pauly,9:1 wet weight to carbon conversion
3,sustainable_catch_global,93.0000,million tonnes/yr,Gulland 1971 / FAO,Maximum sustainable yield for global marine ca...
4,continental_shelf_area,"2,000,000,000.0000",hectares,Various oceanographic,~2 billion ha of continental shelf



CARBON SEQUESTRATION PARAMETERS


,parameter,value,uncertainty,unit,source,notes
0,AFCS,0.7300,± 0.37,tonnes C / ha / year,Mancini et al. (NFA 2016),Average Forest-Carbon Sequestration rate for w...
1,ocean_co2_fraction_min,0.2800,NaN,fraction,Khatiwala et al. 2009 (Nature),Minimum ocean absorption of anthropogenic CO2 ...
2,ocean_co2_fraction_max,0.3500,NaN,fraction,Khatiwala et al. 2009 (Nature),Maximum ocean absorption of anthropogenic CO2 ...
3,ocean_co2_fraction_default,0.3500,NaN,fraction,Wackernagel et al. 2002 / IPCC 2001,Commonly used constant for ocean CO2 absorptio...



FOREST NAI REFERENCE


,parameter,value,unit,source,notes
0,world_avg_forest_NAI,1.8100,m3 harvestable wood / ha / year,UNECE/FAO TBFRA 2000 + FAO GFSM 1998,Net Annual Increment. Denominator for forest p...


In [3]:
# ── All constants used in calculate_overshoot.py ────────────────────────────
EQF_CROPLAND = 2.1
EQF_BUILT_UP = 2.2
EQF_FOREST   = 1.3
EQF_GRAZING  = 0.5
EQF_FISHING  = 0.4
EQF_CARBON   = 1.3

GRASSLAND_NPP = 2.45
FOREST_NAI    = 1.81
AFCS          = 0.73
OCEAN_CO2_FRAC = 0.35

TRANSFER_EFFICIENCY = 0.1013
DISCARD_RATE        = 1.27
WET_TO_CARBON       = 9.0
SUSTAINABLE_CATCH_MT = 93.0
CONTINENTAL_SHELF_HA = 2_000_000_000

TROPHIC_LEVELS = {
    "Freshwater Fish": 2.5, "Demersal Fish": 3.8, "Pelagic Fish": 3.0,
    "Marine Fish, Other": 3.2, "Crustaceans": 2.5, "Cephalopods": 3.5,
    "Molluscs, Other": 2.1, "Aquatic Animals, Others": 3.0,
    "Aquatic Products, Other": 3.0, "Meat, Aquatic Mammals": 3.2,
}

GLEAM_PASTURE = {
    "Cattle": 2.35, "Buffalo": 2.409, "Sheep": 0.46,
    "Goats": 0.394, "Camels": 2.0, "Swine / pigs": 0.0,
}

constants_summary = pd.DataFrame([
    ["EQF_CROPLAND", 2.1, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_BUILT_UP", 2.2, "gha/ha", "Assumes cropland productivity"],
    ["EQF_FOREST", 1.3, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_GRAZING", 0.5, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_FISHING", 0.4, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_CARBON", 1.3, "gha/ha", "Uses forest EQF"],
    ["GRASSLAND_NPP", 2.45, "t DM/ha/yr", "Monfreda et al. 2008"],
    ["FOREST_NAI", 1.81, "m\u00b3/ha/yr", "UNECE/FAO TBFRA 2000"],
    ["AFCS", 0.73, "t C/ha/yr", "Mancini et al. 2016"],
    ["OCEAN_CO2_FRAC", 0.35, "fraction", "Khatiwala et al. 2009"],
    ["TRANSFER_EFFICIENCY", 0.1013, "fraction", "Pauly & Christensen 1995"],
    ["DISCARD_RATE", 1.27, "ratio", "Pauly & Christensen 1995"],
    ["WET_TO_CARBON", 9.0, "ratio", "Sea Around Us / Pauly"],
    ["SUSTAINABLE_CATCH", 93.0, "Mt/yr", "Gulland 1971 / FAO"],
    ["CONTINENTAL_SHELF", 2e9, "ha", "Various oceanographic"],
], columns=["Constant", "Value", "Unit", "Source"])

print("\n" + "=" * 70)
print("ALL HARD-CODED CONSTANTS")
print("=" * 70)
display(constants_summary)


ALL HARD-CODED CONSTANTS


,Constant,Value,Unit,Source
0,EQF_CROPLAND,2.1000,gha/ha,Wackernagel et al. 2002
1,EQF_BUILT_UP,2.2000,gha/ha,Assumes cropland productivity
2,EQF_FOREST,1.3000,gha/ha,Wackernagel et al. 2002
3,EQF_GRAZING,0.5000,gha/ha,Wackernagel et al. 2002
4,EQF_FISHING,0.4000,gha/ha,Wackernagel et al. 2002
5,EQF_CARBON,1.3000,gha/ha,Uses forest EQF
6,GRASSLAND_NPP,2.4500,t DM/ha/yr,Monfreda et al. 2008
7,FOREST_NAI,1.8100,m³/ha/yr,UNECE/FAO TBFRA 2000
8,AFCS,0.7300,t C/ha/yr,Mancini et al. 2016
9,OCEAN_CO2_FRAC,0.3500,fraction,Khatiwala et al. 2009


---
## Section 2: Input Data Overview

All input data comes from the **FAO Statistical Database (FAOSTAT)**, the UN Food and Agriculture
Organization's open data platform covering agriculture, land use, trade, and emissions statistics
for 245+ countries from 1961 to 2023.

For each of the 8 FAOSTAT input files: shape, columns, year range, and sample rows filtered to USA 2023.

**Key filter:** `Area Code < 5000` selects individual countries. Codes ≥ 5000 are FAOSTAT aggregate
regions (e.g., "World", "Africa", "Eastern Europe") and must be excluded to avoid double-counting
when computing world totals by summing across countries.

**Legend: Input Data Files**

| File | FAOSTAT Domain | Content | Unit |
|------|---------------|---------|------|
| `crop_production.csv` | Production (QCL) | Crop production, area harvested, yield | tonnes, ha |
| `cropland_area.csv` | Land Use (RL) | Arable land + permanent crops area | 1000 ha |
| `livestock_stocks.csv` | Production (QCL) | Animal stock counts by species | head |
| `pasture_area.csv` | Land Use (RL) | Permanent meadows & pastures | 1000 ha |
| `timber_production.csv` | Forestry (FO) | Roundwood production | m³ |
| `forest_area.csv` | Land Use (RL) | Forest land area | 1000 ha |
| `fish_supply_fbs.csv` | Food Balance Sheets | Fish production by category | 1000 tonnes |
| `co2_energy_faostat.csv` | Emissions Totals (GT) | CO₂ from energy sector | kilotonnes |

In [4]:
datasets = [
    ("crop_production.csv", crop_prod),
    ("cropland_area.csv", cropland_area),
    ("livestock_stocks.csv", livestock),
    ("pasture_area.csv", pasture_area),
    ("timber_production.csv", timber),
    ("forest_area.csv", forest_area),
    ("fish_supply_fbs.csv", fish_fbs),
    ("co2_energy_faostat.csv", co2_energy),
]

for name, df in datasets:
    print(f"\n{'=' * 70}")
    print(f"{name}")
    print(f"{'=' * 70}")
    print(f"Shape: {df.shape}  |  Columns: {list(df.columns)}")
    print(f"Years: {df['Year'].min()}-{df['Year'].max()}  |  Countries (Area Code < 5000): {df[df['Area Code'] < 5000]['Area Code'].nunique()}")
    usa = df[(df['Area Code'] == DEMO_COUNTRY) & (df['Year'] == DEMO_YEAR)]
    print(f"USA 2023 rows: {len(usa)}")
    display(usa.head(3))


crop_production.csv
Shape: (26273, 14)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']
Years: 2014-2023  |  Countries (Area Code < 5000): 198
USA 2023 rows: 12


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
128387,231,'840,United States of America,1717,'F1717,"Cereals, primary",5312,Area harvested,2023,2023,ha,"55,537,673.0000",A,NaN
128451,231,'840,United States of America,1717,'F1717,"Cereals, primary",5510,Production,2023,2023,t,"462,631,157.5500",A,NaN
128515,231,'840,United States of America,1738,'F1738,Fruit Primary,5312,Area harvested,2023,2023,ha,"1,039,118.0000",A,NaN



cropland_area.csv
Shape: (7695, 13)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


Years: 2014-2023  |  Countries (Area Code < 5000): 228
USA 2023 rows: 3


,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
36651,231,'840,United States of America,6610,Agricultural land,5110,Area,2023,2023,1000 ha,"421,540.3674",I,NaN
36714,231,'840,United States of America,6621,Arable land,5110,Area,2023,2023,1000 ha,"151,563.5251",I,NaN
36777,231,'840,United States of America,6650,Permanent crops,5110,Area,2023,2023,1000 ha,"3,170.6000",I,NaN


...


fish_supply_fbs.csv
Shape: (318114, 14)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (FBS)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


Years: 2014-2023  |  Countries (Area Code < 5000): 179
USA 2023 rows: 184


,Area Code,Area Code (M49),Area,Item Code,Item Code (FBS),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
136332,231,'840,United States of America,2781,'S2781,"Fish, Body Oil",5511,Production,2023,2023,1000 t,85.0000,E,NaN
137606,231,'840,United States of America,2781,'S2781,"Fish, Body Oil",5611,Import quantity,2023,2023,1000 t,20.0000,E,NaN
138864,231,'840,United States of America,2781,'S2781,"Fish, Body Oil",5072,Stock Variation,2023,2023,1000 t,0.0000,E,NaN



co2_energy_faostat.csv
Shape: (2430, 15)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Source Code', 'Source', 'Unit', 'Value', 'Flag', 'Note']
Years: 2014-2023  |  Countries (Area Code < 5000): 205
USA 2023 rows: 1


,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Year Code,Year,Source Code,Source,Unit,Value,Flag,Note
11612,231,'840,United States of America,6821,Energy,7273,Emissions (CO2),2023,2023,3050,FAO TIER 1,kt,"4,640,000.0000",E,NaN


---
## Section 3: Cropland EF — Step by Step

### Formula

$$\text{EF}_{\text{cropland}} = \sum_{\text{crops}} \frac{P_i}{Y_{W,i}} \times \text{EQF}_{\text{cropland}}$$

This is the direct application of Borucke et al. (2013, Eq. 1): for each crop $i$, divide national
production $P_i$ (tonnes) by the world-average yield $Y_{W,i}$ (tonnes/ha) to obtain the number of
world-average hectares needed to produce that quantity. Summing across all crops gives total cropland
demand in world-average hectares, which is then converted to global hectares via the EQF.

> *"The accounts include over 70 crops and 15 secondary products, and the quantity of each product
> allocated to feed, seed, food, waste, processing, and non-food uses."*
> — Monfreda et al. (2004)

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| $P_i$ | varies | National production of crop $i$ (tonnes) | FAOSTAT Production domain, Element = "Production" |
| $Y_{W,i}$ | varies | World yield = Σ(all countries' production) / Σ(all countries' harvested area) | Computed from same FAOSTAT data, Area Code < 5000 |
| EQF | **2.1** | Cropland equivalence factor (gha/ha) | Wackernagel et al. (2002), Table 1; Monfreda et al. (2004) |

### Why world yield?

Using world-average yield (rather than national yield) means we measure how much *globally average*
cropland would be needed. A country with high agricultural productivity "occupies" less world-average
area per tonne than one with low productivity. This is how the EF captures the demand-side signal
regardless of local efficiency (Galli et al. 2007).

### Calculation Steps

1. Filter `crop_production.csv` for USA 2023, Element = "Production"
2. Compute world yield per crop: sum production across all countries / sum harvested area across all countries
3. Divide each crop's national production by its world yield → hectare-equivalents
4. Sum all hectare-equivalents, multiply by EQF (2.1) → final EF in gha

In [5]:
# Step 1: USA 2023 crop production (top 10 by production tonnes)
usa_prod = crop_prod[
    (crop_prod["Area Code"] == DEMO_COUNTRY)
    & (crop_prod["Year"] == DEMO_YEAR)
    & (crop_prod["Element"] == "Production")
].copy()
usa_prod = usa_prod.sort_values("Value", ascending=False)

print(f"USA 2023: {len(usa_prod)} crop items with production data")
print(f"\nTop 10 crops by production (tonnes):")
display(usa_prod[["Item Code", "Item", "Value", "Unit"]].head(10))

USA 2023: 6 crop items with production data

Top 10 crops by production (tonnes):


,Item Code,Item,Value,Unit
128451,1717,"Cereals, primary","462,631,157.5500",t
129091,1735,Vegetables Primary,"31,253,511.2700",t
128707,1732,"Oilcrops, Oil Equivalent","23,240,551.3100",t
128579,1738,Fruit Primary,"21,876,188.6900",t
128963,1720,"Roots and Tubers, Total","21,131,470.4800",t
128835,1726,"Pulses, Total","2,367,610.0000",t


In [6]:
# Step 2: Compute world yield per crop (world production / world harvested area)
_cp = crop_prod[crop_prod["Area Code"] < 5000].copy()
_cp_prod = (
    _cp[_cp["Element"] == "Production"]
    .groupby(["Item Code", "Year"])["Value"].sum()
    .rename("world_production_t")
)
_cp_area = (
    _cp[_cp["Element"] == "Area harvested"]
    .groupby(["Item Code", "Year"])["Value"].sum()
    .rename("world_area_ha")
)
world_crop_yield = pd.concat([_cp_prod, _cp_area], axis=1)
world_crop_yield["world_yield_t_per_ha"] = (
    world_crop_yield["world_production_t"] / world_crop_yield["world_area_ha"]
)
world_crop_yield["world_yield_t_per_ha"] = (
    world_crop_yield["world_yield_t_per_ha"].replace([np.inf, -np.inf], 0).fillna(0)
)

# Show world yield for USA's top 10 crops
top10_items = usa_prod["Item Code"].head(10).tolist()
wy_2023 = world_crop_yield.loc[
    world_crop_yield.index.get_level_values("Year") == DEMO_YEAR
].copy()
wy_top10 = wy_2023.loc[wy_2023.index.get_level_values("Item Code").isin(top10_items)]

print("World yield for USA's top 10 crops (2023):")
display(wy_top10)

World yield for USA's top 10 crops (2023):


,,world_production_t,world_area_ha,world_yield_t_per_ha
Item Code,Year,,,
1717,2023,"3,768,503,439.2700","839,332,298.0000",4.4899
1720,2023,"1,085,273,164.8100","78,152,798.0000",13.8866
1726,2023,"98,505,370.4300","92,205,777.0000",1.0683
1732,2023,"263,010,876.8900","358,416,770.0000",0.7338
1735,2023,"1,781,927,135.9900","83,139,439.0000",21.4330
1738,2023,"1,218,679,287.5300","83,799,849.0000",14.5427


In [7]:
# Step 3: Compute ha-equivalents for each crop
usa_prod_merged = usa_prod.merge(
    world_crop_yield[["world_yield_t_per_ha"]].reset_index(),
    on=["Item Code", "Year"],
    how="left",
)
usa_prod_merged["ha_equivalent"] = np.where(
    usa_prod_merged["world_yield_t_per_ha"] > 0,
    usa_prod_merged["Value"] / usa_prod_merged["world_yield_t_per_ha"],
    0,
)

print("Top 10 crops: production, world yield, ha-equivalents:")
display(
    usa_prod_merged[["Item", "Value", "world_yield_t_per_ha", "ha_equivalent"]]
    .head(10)
    .rename(columns={"Value": "production_t"})
)

Top 10 crops: production, world yield, ha-equivalents:


,Item,production_t,world_yield_t_per_ha,ha_equivalent
0,"Cereals, primary","462,631,157.5500",4.4899,"103,038,587.8242"
1,Vegetables Primary,"31,253,511.2700",21.4330,"1,458,196.2086"
2,"Oilcrops, Oil Equivalent","23,240,551.3100",0.7338,"31,670,946.2059"
3,Fruit Primary,"21,876,188.6900",14.5427,"1,504,268.8652"
4,"Roots and Tubers, Total","21,131,470.4800",13.8866,"1,521,721.5328"
5,"Pulses, Total","2,367,610.0000",1.0683,"2,216,197.1345"


In [8]:
# Step 4: Sum and multiply by EQF
total_ha = usa_prod_merged["ha_equivalent"].sum()
ef_cropland = total_ha * EQF_CROPLAND

stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_cropland_gha")

print(f"Total ha-equivalents across all {len(usa_prod_merged)} crops: {total_ha:,.0f} ha")
print(f"EF_cropland = {total_ha:,.0f} x {EQF_CROPLAND} = {ef_cropland:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_cropland:,.0f} gha")
print(f"Match: {abs(ef_cropland - stored_val) / stored_val < 0.001}")

Total ha-equivalents across all 6 crops: 141,409,918 ha
EF_cropland = 141,409,918 x 2.1 = 296,960,827 gha

Stored result:  296,960,827 gha
Computed here:  296,960,827 gha
Match: True


---
## Section 4: Cropland BC — Step by Step

### Formula

$$\text{BC}_{\text{cropland}} = A_N \times \text{YF} \times \text{EQF}_{\text{cropland}}$$

This follows Borucke et al. (2013, Eq. 5): national biocapacity equals the physical area available
($A_N$), scaled by the **Yield Factor** (how productive national land is relative to world average)
and the **Equivalence Factor** (how productive cropland is relative to all land types).

### Yield Factor (YF) — Concept

> *"If an average Italian hectare of cropland annually produces 6 tonnes of wheat while the
> world-average productivity of wheat is 2 t/ha/year, the yield factor for wheat production
> equals 3. This means that demanding 1 hectare of Italian cropland (nha) is equal to using
> 3 world-average hectares of cropland (wha)."*
> — Galli et al. (2007), p. 181

The YF bridges between physical hectares (national) and world-average hectares:

$$\text{YF} = \frac{Y_N}{Y_W}$$

where $Y_N$ is the aggregate national yield and $Y_W$ is the aggregate world yield across all crops.
A YF > 1 means the country's cropland is more productive than the world average; YF < 1 means less.
We clip to [0.01, 10.0] to prevent extreme outliers from small island nations.

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| $A_N$ | varies | Arable land (Item 6621) + Permanent crops (Item 6650), in ha | FAOSTAT Land Use domain (RL) |
| YF | computed | National yield / world yield (weighted across all crops) | Computed from FAOSTAT Production data |
| EQF | **2.1** | Cropland equivalence factor (gha/ha) | Wackernagel et al. (2002) |

### Calculation Steps

1. Sum national cropland area (Arable land + Permanent crops) from `cropland_area.csv`, convert 1000 ha → ha
2. Compute national aggregate yield = total production / total harvested area
3. Compute world aggregate yield from the world crop yield table
4. YF = national yield / world yield, clipped to [0.01, 10.0]
5. BC = area × YF × EQF

In [9]:
# Step 1: Get cropland area for USA 2023
usa_cla = cropland_area[
    (cropland_area["Area Code"] == DEMO_COUNTRY)
    & (cropland_area["Year"] == DEMO_YEAR)
    & (cropland_area["Item Code"].isin([6621, 6650]))
]
print("USA 2023 cropland area:")
display(usa_cla[["Item Code", "Item", "Value", "Unit"]])

area_1000ha = usa_cla["Value"].sum()
area_ha = area_1000ha * 1000
print(f"\nTotal cropland: {area_1000ha:,.1f} thousand ha = {area_ha:,.0f} ha")

USA 2023 cropland area:


,Item Code,Item,Value,Unit
36714,6621,Arable land,"151,563.5251",1000 ha
36777,6650,Permanent crops,"3,170.6000",1000 ha



Total cropland: 154,734.1 thousand ha = 154,734,125 ha


In [10]:
# Step 2: Compute national yield (aggregate across all crops)
nat_prod_rows = crop_prod[
    (crop_prod["Area Code"] == DEMO_COUNTRY)
    & (crop_prod["Year"] == DEMO_YEAR)
    & (crop_prod["Element"] == "Production")
]
nat_area_rows = crop_prod[
    (crop_prod["Area Code"] == DEMO_COUNTRY)
    & (crop_prod["Year"] == DEMO_YEAR)
    & (crop_prod["Element"] == "Area harvested")
]
nat_prod_total = nat_prod_rows["Value"].sum()
nat_area_total = nat_area_rows["Value"].sum()
nat_yield = nat_prod_total / nat_area_total

print(f"USA national production total: {nat_prod_total:,.0f} t")
print(f"USA national harvested area:   {nat_area_total:,.0f} ha")
print(f"USA national yield:            {nat_yield:,.4f} t/ha")

USA national production total: 562,500,489 t
USA national harvested area:   97,530,289 ha
USA national yield:            5.7674 t/ha


In [11]:
# Step 3: Compute world yield (aggregate across all crops, all countries)
wld = world_crop_yield.loc[world_crop_yield.index.get_level_values("Year") == DEMO_YEAR]
world_prod_total = wld["world_production_t"].sum()
world_area_total = wld["world_area_ha"].sum()
world_yield = world_prod_total / world_area_total

print(f"World production total: {world_prod_total:,.0f} t")
print(f"World harvested area:   {world_area_total:,.0f} ha")
print(f"World average yield:    {world_yield:,.4f} t/ha")

World production total: 8,215,899,275 t
World harvested area:   1,535,046,931 ha
World average yield:    5.3522 t/ha


In [12]:
# Step 4: YF and final BC
yf = nat_yield / world_yield
yf_clipped = np.clip(yf, 0.01, 10.0)

bc_cropland = area_ha * yf_clipped * EQF_CROPLAND
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_cropland_gha")

print(f"Yield Factor (YF) = {nat_yield:,.4f} / {world_yield:,.4f} = {yf:,.4f}")
print(f"YF clipped [0.01, 10.0]: {yf_clipped:,.4f}")
print(f"\nBC_cropland = {area_ha:,.0f} x {yf_clipped:,.4f} x {EQF_CROPLAND} = {bc_cropland:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_cropland:,.0f} gha")
print(f"Match: {abs(bc_cropland - stored_val) / stored_val < 0.001}")

Yield Factor (YF) = 5.7674 / 5.3522 = 1.0776
YF clipped [0.01, 10.0]: 1.0776

BC_cropland = 154,734,125 x 1.0776 x 2.1 = 350,150,978 gha

Stored result:  350,150,978 gha
Computed here:  350,150,978 gha
Match: True


---
## Section 5: Grazing EF — Step by Step

### Formula

$$\text{EF}_{\text{grazing}} = \frac{\sum_j (\text{Heads}_j \times \text{PastureDM}_j)}{\text{NPP}_{\text{grassland}}} \times \text{EQF}_{\text{grazing}}$$

The grazing footprint converts livestock populations into the pasture area needed to feed them.
Borucke et al. (2013, Eq. 10) describe the full methodology: total feed requirement minus
marketed crops, fodder crops, and crop residues yields the *pasture grass requirement* (PGR).

In this simplified implementation, we use **GLEAM coefficients** (FAO Global Livestock Environmental
Assessment Model, version 3.0) to directly estimate the annual pasture dry matter demand per head
for each livestock species, then divide by the grassland net primary productivity to obtain the
area equivalent.

> *"National pasture productivities are estimated from tropical, temperate, and arid grassland
> primary production data."*
> — Monfreda et al. (2004)

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| Heads$_j$ | varies | National livestock stock count for species $j$ | FAOSTAT Production domain (QCL), Element = "Stocks" |
| PastureDM$_j$ | see table below | Annual pasture dry matter demand (t DM/head/yr) | FAO GLEAM 3.0 global averages |
| NPP | **2.45** t DM/ha/yr | World-average above-ground edible grassland NPP | Monfreda et al. (2008), *Global Biogeochem. Cycles* 22(1) |
| EQF | **0.5** | Grazing land equivalence factor (gha/ha) | Wackernagel et al. (2002) |

### GLEAM Pasture Demand Coefficients

| Livestock | Pasture DM (t/head/yr) | Notes |
|-----------|:---------------------:|-------|
| Cattle | 2.35 | Average of dairy (2.382) and beef (2.325) |
| Buffalo | 2.409 | |
| Sheep | 0.46 | |
| Goats | 0.394 | |
| Camels | 2.0 | Approximate, similar to cattle |
| Swine / pigs | 0.0 | Zero pasture (fed on crops/residues) |

### Interpretation

The denominator (NPP = 2.45 t DM/ha/yr) represents the annual harvestable biomass produced by one
hectare of world-average grassland. Dividing total feed demand by NPP gives the grassland area that
would be needed at world-average productivity. Multiplying by EQF (0.5) converts to gha — reflecting
that grassland is only half as productive as the average hectare of all bioproductive land.

### Calculation Steps

1. Get livestock stock counts for USA 2023
2. Map each animal type to its GLEAM pasture demand coefficient
3. Multiply heads × demand → total pasture demand in tonnes DM
4. Divide by NPP (2.45), multiply by EQF (0.5) → EF in gha

In [13]:
# Step 1: USA 2023 livestock stocks
usa_ls = livestock[
    (livestock["Area Code"] == DEMO_COUNTRY) & (livestock["Year"] == DEMO_YEAR)
].copy()
print("USA 2023 livestock stocks:")
display(usa_ls[["Item", "Value", "Unit"]])

USA 2023 livestock stocks:


,Item,Value,Unit
47629,Cattle,"88,841,000.0000",An
47693,Goats,"2,527,000.0000",An
47757,Sheep,"5,130,000.0000",An
47821,Swine / pigs,"75,461,300.0000",An


In [14]:
# Step 2: GLEAM pasture demand mapping
gleam_table = pd.DataFrame(
    list(GLEAM_PASTURE.items()),
    columns=["FAOSTAT Item", "Pasture DM (t/head/yr)"]
)
print("GLEAM pasture demand mapping (as used in code):")
display(gleam_table)

GLEAM pasture demand mapping (as used in code):


,FAOSTAT Item,Pasture DM (t/head/yr)
0,Cattle,2.3500
1,Buffalo,2.4090
2,Sheep,0.4600
3,Goats,0.3940
4,Camels,2.0000
5,Swine / pigs,0.0000


In [15]:
# Step 3: Compute pasture demand per animal type
usa_ls["pasture_demand_per_head"] = usa_ls["Item"].map(GLEAM_PASTURE).fillna(0)
usa_ls["total_pasture_demand_t"] = usa_ls["Value"].fillna(0) * usa_ls["pasture_demand_per_head"]

print("Pasture demand by livestock type:")
display(
    usa_ls[["Item", "Value", "pasture_demand_per_head", "total_pasture_demand_t"]]
    .rename(columns={"Value": "heads"})
)

total_demand = usa_ls["total_pasture_demand_t"].sum()
print(f"\nTotal pasture demand: {total_demand:,.0f} tonnes DM")

Pasture demand by livestock type:


,Item,heads,pasture_demand_per_head,total_pasture_demand_t
47629,Cattle,"88,841,000.0000",2.3500,"208,776,350.0000"
47693,Goats,"2,527,000.0000",0.3940,"995,638.0000"
47757,Sheep,"5,130,000.0000",0.4600,"2,359,800.0000"
47821,Swine / pigs,"75,461,300.0000",0.0000,0.0000



Total pasture demand: 212,131,788 tonnes DM


In [16]:
# Step 4: Divide by NPP, multiply by EQF
grazing_ha = total_demand / GRASSLAND_NPP
ef_grazing = grazing_ha * EQF_GRAZING
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_grazing_gha")

print(f"Grazing area = {total_demand:,.0f} / {GRASSLAND_NPP} = {grazing_ha:,.0f} ha")
print(f"EF_grazing = {grazing_ha:,.0f} x {EQF_GRAZING} = {ef_grazing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_grazing:,.0f} gha")
print(f"Match: {abs(ef_grazing - stored_val) / stored_val < 0.001}")

Grazing area = 212,131,788 / 2.45 = 86,584,403 ha
EF_grazing = 86,584,403 x 0.5 = 43,292,202 gha

Stored result:  43,292,202 gha
Computed here:  43,292,202 gha
Match: True


---
## Section 6: Grazing BC — Step by Step

### Formula

$$\text{BC}_{\text{grazing}} = A_{\text{pasture}} \times \text{YF} \times \text{EQF}_{\text{grazing}}$$

Grazing biocapacity follows the same general BC formula (Borucke et al. 2013, Eq. 5).

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| $A_{\text{pasture}}$ | varies | Permanent meadows & pastures, in ha | FAOSTAT Land Use (RL), converted from 1000 ha |
| YF | **1.0** | Yield factor (simplified — no country-specific adjustment) | See note below |
| EQF | **0.5** | Grazing land equivalence factor (gha/ha) | Wackernagel et al. (2002) |

### Note on YF = 1.0

In the official NFA, grazing YF varies by country based on national grassland productivity relative
to the world average (Borucke et al. 2013, §3.3). This implementation uses YF = 1.0 for simplicity,
which assumes all countries' grasslands are equally productive. This is a known simplification
(see README, "Limitations and known differences from GFN").

### Calculation Steps

1. Get pasture area from `pasture_area.csv` for USA 2023 (in 1000 ha → multiply by 1000)
2. BC = area × 1.0 × EQF (0.5)

In [17]:
# USA 2023 pasture area
usa_pa = pasture_area[
    (pasture_area["Area Code"] == DEMO_COUNTRY) & (pasture_area["Year"] == DEMO_YEAR)
]
print("USA 2023 pasture area:")
display(usa_pa[["Item", "Value", "Unit"]])

pa_1000ha = usa_pa["Value"].sum()
pa_ha = pa_1000ha * 1000
bc_grazing = pa_ha * 1.0 * EQF_GRAZING
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_grazing_gha")

print(f"\nPasture area: {pa_1000ha:,.1f} thousand ha = {pa_ha:,.0f} ha")
print(f"BC_grazing = {pa_ha:,.0f} x 1.0 x {EQF_GRAZING} = {bc_grazing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_grazing:,.0f} gha")
print(f"Match: {abs(bc_grazing - stored_val) / stored_val < 0.001}")

USA 2023 pasture area:


,Item,Value,Unit
11724,Permanent meadows and pastures,"266,806.2423",1000 ha



Pasture area: 266,806.2 thousand ha = 266,806,242 ha
BC_grazing = 266,806,242 x 1.0 x 0.5 = 133,403,121 gha

Stored result:  133,403,121 gha
Computed here:  133,403,121 gha
Match: True


---
## Section 7: Forest EF — Step by Step

### Formula

$$\text{EF}_{\text{forest}} = \frac{\text{Roundwood}_{m^3}}{\text{NAI}} \times \text{EQF}_{\text{forest}}$$

The forest footprint translates roundwood production into the forest area required to sustain that
harvest at the world-average growth rate. The denominator — the **Net Annual Increment** (NAI) —
represents how many cubic metres of merchantable timber one hectare of world-average forest grows
per year (Borucke et al. 2013, §3.4).

> *"The yield used in the forest land Footprint is the net annual increment (NAI) of merchantable
> timber per hectare."*
> — Borucke et al. (2013)

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| Roundwood | varies | National roundwood production (m³) | FAOSTAT Forestry domain (FO), Item = "Roundwood", Element = "Production" |
| NAI | **1.81** m³/ha/yr | World-average net annual increment of harvestable wood | UNECE/FAO Temperate and Boreal Forest Resource Assessment (TBFRA) 2000; FAO Global Fiber Supply Model 1998 |
| EQF | **1.3** | Forest equivalence factor (gha/ha) | Wackernagel et al. (2002) |

### Context

Roundwood is the aggregate of all wood removals — industrial roundwood (sawlogs, pulpwood, veneer logs)
plus wood fuel (firewood, charcoal). The official NFA tracks 13 timber products and 3 wood fuel products
as primary forest products, plus 30 timber trade products (Borucke et al. 2013, §3.4). This simplified
implementation uses only the aggregate "Roundwood" production figure from FAOSTAT.

### Calculation Steps

1. Get roundwood production for USA 2023 from `timber_production.csv`
2. Divide by NAI (1.81 m³/ha/yr) → forest area in ha
3. Multiply by EQF (1.3) → EF in gha

In [18]:
# USA 2023 roundwood production
usa_timber = timber[
    (timber["Area Code"] == DEMO_COUNTRY)
    & (timber["Year"] == DEMO_YEAR)
    & (timber["Item"] == "Roundwood")
    & (timber["Element"] == "Production")
]
print("USA 2023 timber production:")
display(usa_timber[["Item", "Element", "Value", "Unit"]])

roundwood_m3 = usa_timber["Value"].sum()
print(f"\nRoundwood production: {roundwood_m3:,.0f} m\u00b3")

USA 2023 timber production:


,Item,Element,Value,Unit
95804,Roundwood,Production,"389,026,299.0000",m3



Roundwood production: 389,026,299 m³


In [19]:
# Compute EF
forest_ha = roundwood_m3 / FOREST_NAI
ef_forest = forest_ha * EQF_FOREST
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_forest_gha")

print(f"Forest area needed = {roundwood_m3:,.0f} / {FOREST_NAI} = {forest_ha:,.0f} ha")
print(f"EF_forest = {forest_ha:,.0f} x {EQF_FOREST} = {ef_forest:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_forest:,.0f} gha")
print(f"Match: {abs(ef_forest - stored_val) / stored_val < 0.001}")

Forest area needed = 389,026,299 / 1.81 = 214,931,657 ha
EF_forest = 214,931,657 x 1.3 = 279,411,154 gha

Stored result:  279,411,154 gha
Computed here:  279,411,154 gha
Match: True


---
## Section 8: Forest BC — Step by Step

### Formula

$$\text{BC}_{\text{forest}} = A_{\text{forest}} \times \text{YF} \times \text{EQF}_{\text{forest}}$$

Forest biocapacity represents the total regenerative capacity of a country's forests, measured in
global hectares. This includes the capacity to both produce timber *and* sequester carbon — the
carbon footprint (Section 12) draws on this same biocapacity pool.

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| $A_{\text{forest}}$ | varies | National forest area, in ha | FAOSTAT Land Use (RL), converted from 1000 ha |
| YF | **1.0** | Yield factor (simplified — no country-specific adjustment) | See note in Section 6 |
| EQF | **1.3** | Forest equivalence factor (gha/ha) | Wackernagel et al. (2002) |

### Relationship to Carbon Footprint

> *"The uptake land to accommodate the carbon Footprint is the only land use type included in the
> NFAs that is exclusively dedicated to tracking a waste product: carbon dioxide."*
> — Borucke et al. (2013)

Forest BC covers both timber production capacity and CO₂ sequestration capacity.
The carbon EF (Section 12) has no separate biocapacity — it competes with the forest product EF
for the same forest biocapacity. When a country's combined forest EF + carbon EF exceeds its forest BC,
it is in deficit for this land type.

### Calculation Steps

1. Get forest area from `forest_area.csv` for USA 2023 (in 1000 ha → multiply by 1000)
2. BC = area × 1.0 × EQF (1.3)

In [20]:
# USA 2023 forest area
usa_fa = forest_area[
    (forest_area["Area Code"] == DEMO_COUNTRY) & (forest_area["Year"] == DEMO_YEAR)
]
print("USA 2023 forest area:")
display(usa_fa[["Item", "Value", "Unit"]])

fa_1000ha = usa_fa["Value"].sum()
fa_ha = fa_1000ha * 1000
bc_forest = fa_ha * 1.0 * EQF_FOREST
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_forest_gha")

print(f"\nForest area: {fa_1000ha:,.1f} thousand ha = {fa_ha:,.0f} ha")
print(f"BC_forest = {fa_ha:,.0f} x 1.0 x {EQF_FOREST} = {bc_forest:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_forest:,.0f} gha")
print(f"Match: {abs(bc_forest - stored_val) / stored_val < 0.001}")

USA 2023 forest area:


,Item,Value,Unit
14723,Forest land,"309,039.0000",1000 ha
14759,Forest land,33.7800,%



Forest area: 309,072.8 thousand ha = 309,072,780 ha
BC_forest = 309,072,780 x 1.0 x 1.3 = 401,794,614 gha

Stored result:  401,794,614 gha
Computed here:  401,794,614 gha
Match: True


---
## Section 9: Fishing EF — Step by Step

### The PPR Method

The fishing grounds footprint uses the **Primary Production Required** (PPR) approach from
Pauly & Christensen (1995) to convert fish catch into the marine area needed to sustain it.

> *"Primary production required to sustain global fisheries"*
> — Pauly, D. & Christensen, V. (1995), *Nature* 374: 255–257

The key insight: higher-trophic-level fish (e.g., tuna at TL ≈ 4.5) require *exponentially* more
phytoplankton production to sustain each tonne of catch than low-trophic-level fish (e.g., anchovies
at TL ≈ 2.5), because energy is lost at each step in the food chain.

### Formulas

**Per fish category** (Borucke et al. 2013, Eq. 11):
$$\text{PPR}_i = \frac{\text{Catch}_{i,t} \times \text{DR}}{WC} \times \left(\frac{1}{\text{TE}}\right)^{\text{TL}_i - 1}$$

**Marine yield** (Borucke et al. 2013, Eq. 12–13):
$$Y_M = \frac{\text{World Sustainable PPR}}{A_{\text{shelf}}}$$

**Final EF:**
$$\text{EF}_{\text{fishing}} = \frac{\sum_i \text{PPR}_i}{Y_M} \times \text{EQF}_{\text{fishing}}$$

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| Catch$_i$ | varies | National production of fish category $i$ (tonnes) | FAOSTAT FBS, Element = "Production" (in 1000 t, ×1000) |
| DR | **1.27** | Discard rate: 1 + 0.27 bycatch per tonne harvested | Pauly & Christensen (1995) |
| TE | **0.1013** | Transfer efficiency: ~10% per trophic level | Pauly & Christensen (1995) |
| TL$_i$ | see table | Trophic level for each FBS fish category | Literature averages (FishBase / Sea Around Us) |
| WC | **9.0** | Wet-weight to carbon conversion ratio (9:1) | Sea Around Us / Pauly |
| Sustainable catch | **93 Mt/yr** | Global maximum sustainable yield | Gulland (1971), *The Fish Resources of the Ocean* |
| $A_{\text{shelf}}$ | **2 × 10⁹ ha** | Global continental shelf area | Various oceanographic estimates |
| EQF | **0.4** | Fishing grounds equivalence factor (gha/ha) | Wackernagel et al. (2002) |

### Trophic Level Mapping

Each FBS fish category is assigned an average trophic level based on species-level data
from FishBase and the Sea Around Us project:

| FBS Category | TL | Ecological role |
|---|:---:|---|
| Freshwater Fish | 2.5 | Mostly herbivores/omnivores |
| Demersal Fish | 3.8 | Bottom-dwelling predators (cod, haddock) |
| Pelagic Fish | 3.0 | Open-water schooling fish (herring, sardines) |
| Marine Fish, Other | 3.2 | Mixed marine species |
| Crustaceans | 2.5 | Shrimp, crabs, lobsters |
| Cephalopods | 3.5 | Squid, octopus (active predators) |
| Molluscs, Other | 2.1 | Filter-feeders (mussels, oysters) |
| Aquatic Animals, Others | 3.0 | Miscellaneous aquatic animals |
| Aquatic Products, Other | 3.0 | Miscellaneous aquatic products |
| Meat, Aquatic Mammals | 3.2 | Whales, seals |

### Interpretation

The $(1/\text{TE})^{\text{TL}-1}$ term is the exponential amplifier. For demersal fish (TL = 3.8):
$(1/0.1013)^{2.8} = 9.874^{2.8} ≈ 6{,}380$ — meaning each tonne of demersal fish harvest
requires ~6,380 times its weight in primary production. This is why high-trophic-level fisheries
have such a large ecological footprint.

### Calculation Steps

1. Get fish production for USA 2023 from `fish_supply_fbs.csv` (Element = "Production")
2. Map each FBS category to its trophic level (skip categories not in the TL table)
3. Compute PPR per category: catch × DR × (1/TE)^(TL−1) / WC
4. Compute world sustainable PPR from 93 Mt at average TL 3.1
5. Marine yield = world sustainable PPR / continental shelf area
6. EF = total PPR / marine yield × EQF (0.4)

In [21]:
# Step 1: USA 2023 fish production from FBS
usa_fish = fish_fbs[
    (fish_fbs["Area Code"] == DEMO_COUNTRY)
    & (fish_fbs["Year"] == DEMO_YEAR)
    & (fish_fbs["Element"] == "Production")
].copy()
print(f"USA 2023 fish production ({len(usa_fish)} items):")
display(usa_fish[["Item", "Value", "Unit"]])

USA 2023 fish production (13 items):


,Item,Value,Unit
136332,"Fish, Body Oil",85.0000,1000 t
147915,"Fish, Liver Oil",1.0000,1000 t
251428,"Fish, Seafood","5,291.0000",1000 t
273431,Freshwater Fish,601.0000,1000 t
292826,Demersal Fish,"2,559.0000",1000 t
310135,Pelagic Fish,838.0000,1000 t
324724,"Marine Fish, Other",330.0000,1000 t
335846,Crustaceans,257.0000,1000 t
344368,Cephalopods,54.0000,1000 t
351765,"Molluscs, Other",646.0000,1000 t


In [22]:
# Step 2: Trophic level mapping
tl_table = pd.DataFrame(
    list(TROPHIC_LEVELS.items()),
    columns=["FBS Fish Category", "Trophic Level"]
)
print("Trophic level mapping:")
display(tl_table)

Trophic level mapping:


,FBS Fish Category,Trophic Level
0,Freshwater Fish,2.5000
1,Demersal Fish,3.8000
2,Pelagic Fish,3.0000
3,"Marine Fish, Other",3.2000
4,Crustaceans,2.5000
5,Cephalopods,3.5000
6,"Molluscs, Other",2.1000
7,"Aquatic Animals, Others",3.0000
8,"Aquatic Products, Other",3.0000
9,"Meat, Aquatic Mammals",3.2000


In [23]:
# Step 3: Compute PPR for each fish category
usa_fish["tl"] = usa_fish["Item"].map(TROPHIC_LEVELS)
usa_fish_valid = usa_fish.dropna(subset=["tl"]).copy()
usa_fish_valid["prod_t"] = usa_fish_valid["Value"].fillna(0) * 1000  # 1000 t -> t
usa_fish_valid["ppr"] = (
    usa_fish_valid["prod_t"]
    * DISCARD_RATE
    * (1.0 / TRANSFER_EFFICIENCY) ** (usa_fish_valid["tl"] - 1)
    / WET_TO_CARBON
)

print("PPR calculation per fish category:")
display(
    usa_fish_valid[["Item", "Value", "prod_t", "tl", "ppr"]]
    .rename(columns={"Value": "prod_kt"})
)

total_ppr = usa_fish_valid["ppr"].sum()
print(f"\nTotal PPR: {total_ppr:,.0f} t C")

PPR calculation per fish category:


,Item,prod_kt,prod_t,tl,ppr
273431,Freshwater Fish,601.0000,"601,000.0000",2.5000,"2,630,398.3018"
292826,Demersal Fish,"2,559.0000","2,559,000.0000",3.8000,"219,748,060.3596"
310135,Pelagic Fish,838.0000,"838,000.0000",3.0000,"11,523,551.2972"
324724,"Marine Fish, Other",330.0000,"330,000.0000",3.2000,"7,173,553.9955"
335846,Crustaceans,257.0000,"257,000.0000",2.5000,"1,124,812.5850"
344368,Cephalopods,54.0000,"54,000.0000",3.5000,"2,333,089.2786"
351765,"Molluscs, Other",646.0000,"646,000.0000",2.1000,"1,131,418.6666"
357964,"Aquatic Animals, Others",5.0000,"5,000.0000",3.0000,"68,756.2727"
363372,"Aquatic Products, Other",3.0000,"3,000.0000",3.0000,"41,253.7636"



Total PPR: 245,774,895 t C


In [24]:
# Step 4: Compute marine yield and EF
avg_tl = 3.1
world_sust_ppr = (
    SUSTAINABLE_CATCH_MT * 1e6
    * DISCARD_RATE
    * (1.0 / TRANSFER_EFFICIENCY) ** (avg_tl - 1)
    / WET_TO_CARBON
)
marine_yield = world_sust_ppr / CONTINENTAL_SHELF_HA

fishing_area_ha = total_ppr / marine_yield
ef_fishing = fishing_area_ha * EQF_FISHING
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_fishing_gha")

print(f"World sustainable PPR: {world_sust_ppr:,.0f} t C")
print(f"Marine yield = {world_sust_ppr:,.0f} / {CONTINENTAL_SHELF_HA:,.0f} = {marine_yield:.6f} t C/ha")
print(f"Fishing area = {total_ppr:,.0f} / {marine_yield:.6f} = {fishing_area_ha:,.0f} ha")
print(f"EF_fishing = {fishing_area_ha:,.0f} x {EQF_FISHING} = {ef_fishing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_fishing:,.0f} gha")
print(f"Match: {abs(ef_fishing - stored_val) / stored_val < 0.001}")

World sustainable PPR: 1,607,919,584 t C
Marine yield = 1,607,919,584 / 2,000,000,000 = 0.803960 t C/ha
Fishing area = 245,774,895 / 0.803960 = 305,705,456 ha
EF_fishing = 305,705,456 x 0.4 = 122,282,182 gha

Stored result:  122,282,182 gha
Computed here:  122,282,182 gha
Match: True


---
## Section 10: Fishing BC — Step by Step

### Formula

$$\text{BC}_{\text{fishing}} = \text{WorldFishingBC} \times \frac{\text{CountryProd}}{\text{WorldProd}}$$

Unlike land-based biocapacity (which is bounded by national territory), marine biocapacity is a
**global commons**. No country "owns" ocean productivity in the same way it owns cropland. The NFA
therefore allocates the total marine biocapacity proportionally based on each country's share of
world fish production (Borucke et al. 2013, §3.2).

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| WorldFishingBC | **800 million gha** | 2 × 10⁹ ha × 0.4 EQF | Continental shelf area × fishing EQF |
| CountryProd | varies | Country's total fish production (tonnes) | FAOSTAT FBS, filtered to trophic-level categories only |
| WorldProd | varies | World total fish production (tonnes) | Sum across all countries (Area Code < 5000) |

### Why production-proportional allocation?

This approach assumes that a country's "claim" on marine biocapacity is proportional to how much
fish it actually produces. Countries with large fishing industries (China, Indonesia, Peru) receive
correspondingly larger shares of the global marine biocapacity.

### Calculation Steps

1. Compute WorldFishingBC = 2 billion ha × EQF (0.4) = 800 million gha
2. Get USA's fish production and world total (using same trophic-level-filtered categories as EF)
3. BC = WorldFishingBC × (USA production / world production)

In [25]:
# World fishing BC
world_fishing_bc = CONTINENTAL_SHELF_HA * EQF_FISHING
print(f"World Fishing BC = {CONTINENTAL_SHELF_HA:,.0f} ha x {EQF_FISHING} = {world_fishing_bc:,.0f} gha")
print(f"                 = {world_fishing_bc / 1e6:,.0f} million gha")

# Country production share (using trophic-level-valid items only, matching the script)
_ff_all = fish_fbs[
    (fish_fbs["Area Code"] < 5000)
    & (fish_fbs["Element"] == "Production")
].copy()
_ff_all["tl"] = _ff_all["Item"].map(TROPHIC_LEVELS)
_ff_all = _ff_all.dropna(subset=["tl"])
_ff_all["prod_t"] = _ff_all["Value"].fillna(0) * 1000

world_prod_t = _ff_all[_ff_all["Year"] == DEMO_YEAR]["prod_t"].sum()
country_prod_t = usa_fish_valid["prod_t"].sum()
share = country_prod_t / world_prod_t

print(f"\nUSA fish production:   {country_prod_t:,.0f} t")
print(f"World fish production:  {world_prod_t:,.0f} t")
print(f"USA share: {share:.6f} ({share * 100:.2f}%)")

World Fishing BC = 2,000,000,000 ha x 0.4 = 800,000,000 gha
                 = 800 million gha



USA fish production:   5,293,000 t
World fish production:  299,110,000 t
USA share: 0.017696 (1.77%)


In [26]:
# Final BC
bc_fishing = world_fishing_bc * share
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_fishing_gha")

print(f"BC_fishing = {world_fishing_bc:,.0f} x {share:.6f} = {bc_fishing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_fishing:,.0f} gha")
print(f"Match: {abs(bc_fishing - stored_val) / stored_val < 0.001}")

BC_fishing = 800,000,000 x 0.017696 = 14,156,665 gha

Stored result:  14,156,665 gha
Computed here:  14,156,665 gha
Match: True


---
## Section 11: Built-up EF/BC — Step by Step

### Formula

$$\text{EF}_{\text{built-up}} = \text{BC}_{\text{built-up}} = A_{\text{built-up}} \times \text{EQF}_{\text{built-up}}$$

Built-up land is unique: **EF equals BC**. A country's built-up area simultaneously represents both
the demand it places on the biosphere and the biocapacity it provides. You cannot "overshoot" your
own infrastructure footprint — the land is simply occupied.

> *"The NFA assumes that built-up land occupies what would previously have been cropland, except in
> cases where clear evidence exists that built-up land does not sit on cropland."*
> — Borucke et al. (2013)

This is why built-up land uses **EQF = 2.2** (close to the cropland EQF of 2.1) — it reflects the
productivity of the cropland it replaces.

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| $A_{\text{built-up}}$ | varies | Estimated built-up area (ha) | Computed from FAOSTAT Land Use items (see 3-tier method) |
| EQF | **2.2** | Built-up land equivalence factor (gha/ha) | Borucke et al. (2013); assumes cropland replacement |

### 3-Tier Estimation for Built-Up Area

Direct built-up area data is sparse in FAOSTAT. The calculator uses a three-tier fallback:

| Tier | Method | Coverage |
|------|--------|----------|
| **1** | FAOSTAT Item 6649 ("Farm buildings & Farmyards") directly | ~36 countries |
| **2** | Land-use residual: Land area (6601) − Agricultural land (6610) − Forest (6646) − Other land (6670) − Inland waters (6680) | Used if positive |
| **3** | 3% of cropland area (proxy) | Fallback for remaining countries |

### FAOSTAT Land Use Items

| Item Code | Item Name | Role in calculation |
|:---------:|-----------|---------------------|
| 6601 | Country area (land) | Total land area |
| 6610 | Agricultural land | Cropland + pasture |
| 6646 | Forest land | Forests and woodlands |
| 6649 | Farm buildings & farmyards | Direct built-up (Tier 1) |
| 6670 | Other land | Barren, desert, etc. |
| 6680 | Inland waters | Lakes, rivers, reservoirs |

### Calculation Steps

1. Retrieve all relevant land-use items for USA 2023
2. Check if Item 6649 is directly available (Tier 1)
3. If not, compute residual from other land-use categories (Tier 2); if negative, use 3% cropland fallback (Tier 3)
4. EF = BC = built-up area × EQF (2.2)

In [27]:
# Step 1: Show all relevant land-use items for USA 2023
items_of_interest = [6601, 6610, 6646, 6649, 6670, 6680]
usa_lu = land_use_all[
    (land_use_all["Area Code"] == DEMO_COUNTRY)
    & (land_use_all["Year"] == DEMO_YEAR)
    & (land_use_all["Item Code"].isin(items_of_interest))
]
print("USA 2023 land-use items:")
display(usa_lu[["Item Code", "Item", "Value", "Unit"]].sort_values("Item Code"))

USA 2023 land-use items:


,Item Code,Item,Value,Unit
332307,6601,Land area,"914,742.0000",1000 ha
332433,6610,Agricultural land,"421,540.3674",1000 ha
332496,6610,Agricultural land,46.0800,%
332559,6610,Agricultural land,895.4600,USD_PPP/ha
333392,6646,Forest land,"309,039.0000",1000 ha
333428,6646,Forest land,"22,800.7440",million t
333464,6646,Forest land,33.7800,%
333358,6649,Farm buildings and Farmyards,"13,231.6592",1000 ha
333678,6670,Other land,"170,930.9734",1000 ha
333741,6680,Inland waters,"68,409.0000",1000 ha


In [28]:
# Step 2: Compute residual and determine built-up area
items_needed = {6601: 0, 6610: 0, 6646: 0, 6670: 0, 6680: 0}
for _, r in usa_lu.iterrows():
    if r["Item Code"] in items_needed:
        items_needed[r["Item Code"]] = r["Value"] if not pd.isna(r["Value"]) else 0

land_area = items_needed[6601]
ag_land = items_needed[6610]
forest_land = items_needed[6646]
other_land = items_needed[6670]
inland_water = items_needed[6680]
residual = land_area - ag_land - forest_land - other_land - inland_water

# Check if 6649 is directly available
direct_6649 = usa_lu[usa_lu["Item Code"] == 6649]["Value"]
has_6649 = len(direct_6649) > 0 and not pd.isna(direct_6649.iloc[0]) if len(direct_6649) > 0 else False

print(f"Land area (6601):       {land_area:>10,.1f} thousand ha")
print(f"Agricultural land (6610): {ag_land:>10,.1f} thousand ha")
print(f"Forest land (6646):     {forest_land:>10,.1f} thousand ha")
print(f"Other land (6670):      {other_land:>10,.1f} thousand ha")
print(f"Inland water (6680):    {inland_water:>10,.1f} thousand ha")
print(f"Residual:               {residual:>10,.1f} thousand ha")
print(f"\nItem 6649 available: {has_6649}")

if has_6649:
    built_up_1000ha = direct_6649.iloc[0]
    print(f"Using item 6649: {built_up_1000ha:,.1f} thousand ha")
elif residual > 0:
    built_up_1000ha = residual
    print(f"Using positive residual: {built_up_1000ha:,.1f} thousand ha")
else:
    built_up_1000ha = (area_1000ha) * 0.03  # cropland area from Section 4
    print(f"Fallback: 3% of cropland = {built_up_1000ha:,.1f} thousand ha")

Land area (6601):        914,742.0 thousand ha
Agricultural land (6610):      895.5 thousand ha
Forest land (6646):           33.8 thousand ha
Other land (6670):       170,931.0 thousand ha
Inland water (6680):      68,409.0 thousand ha
Residual:                674,472.8 thousand ha

Item 6649 available: True
Using item 6649: 13,231.7 thousand ha


In [29]:
# Step 3: Compute EF = BC = built_up_ha x EQF
built_up_ha = built_up_1000ha * 1000
ef_built_up = built_up_ha * EQF_BUILT_UP
bc_built_up = ef_built_up  # EF = BC for built-up
stored_ef = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_built_up_gha")
stored_bc = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_built_up_gha")

print(f"Built-up area: {built_up_1000ha:,.1f} thousand ha = {built_up_ha:,.0f} ha")
print(f"EF_built_up = BC_built_up = {built_up_ha:,.0f} x {EQF_BUILT_UP} = {ef_built_up:,.0f} gha")
print(f"\nStored EF: {stored_ef:,.0f} gha  |  Computed: {ef_built_up:,.0f} gha  |  Match: {abs(ef_built_up - stored_ef) / stored_ef < 0.01}")
print(f"Stored BC: {stored_bc:,.0f} gha  |  Computed: {bc_built_up:,.0f} gha  |  Match: {abs(bc_built_up - stored_bc) / stored_bc < 0.01}")

Built-up area: 13,231.7 thousand ha = 13,231,659 ha
EF_built_up = BC_built_up = 13,231,659 x 2.2 = 29,109,650 gha

Stored EF: 29,109,650 gha  |  Computed: 29,109,650 gha  |  Match: True
Stored BC: 29,109,650 gha  |  Computed: 29,109,650 gha  |  Match: True


---
## Section 12: Carbon EF — Step by Step

### Formula

$$\text{EF}_{\text{carbon}} = \frac{\text{CO}_2 \text{(tonnes)} \times (1 - S_{\text{ocean}})}{\text{AFCS} \times \frac{44}{12}} \times \text{EQF}_{\text{carbon}}$$

The carbon footprint is conceptually different from the other five components: it measures the
**forest area that would be needed to sequester** the CO₂ emissions that are *not* absorbed by the ocean.

> *"The uptake land to accommodate the carbon Footprint is the only land use type included in the
> NFAs that is exclusively dedicated to tracking a waste product: carbon dioxide."*
> — Borucke et al. (2013)

### Step-by-step Conversion

| Step | Operation | Formula | Purpose |
|:----:|-----------|---------|---------|
| 1 | kt → tonnes | CO₂_kt × 1000 | Unit conversion |
| 2 | Land fraction | × (1 − 0.35) | 35% absorbed by oceans → 65% must be sequestered on land |
| 3 | Sequestration rate | ÷ (AFCS × 44/12) | Convert AFCS from t C/ha/yr to t CO₂/ha/yr |
| 4 | EQF | × 1.3 | Convert forest ha to gha |

### Parameters

| Symbol | Value | Meaning | Source |
|--------|-------|---------|--------|
| CO₂ | varies | Territorial CO₂ emissions from energy use (kilotonnes) | FAOSTAT Emissions domain (GT) |
| $S_{\text{ocean}}$ | **0.35** | Fraction of anthropogenic CO₂ absorbed by oceans (28–35% range) | Khatiwala et al. (2009), *Nature* 462: 346–349 |
| AFCS | **0.73** t C/ha/yr | Annual Forest Carbon Sequestration rate for world-average forests | Mancini et al. (2016), *Ecol. Indicators* 61: 390–403 |
| 44/12 | **3.667** | Molecular weight ratio CO₂/C (converts carbon mass to CO₂ mass) | Chemistry |
| EQF | **1.3** | Uses forest EQF (sequestration occurs on forest land) | Wackernagel et al. (2002) |

### Ocean CO₂ Absorption

Khatiwala et al. (2009) reconstructed the history of anthropogenic CO₂ in the ocean and found
a relatively constant fractional uptake of 28–35% over the period 1961–2008 (varying from ~1.0
to ~2.3 Pg C/yr in absolute terms as emissions grew). This calculator uses the upper bound (35%),
following the convention in Wackernagel et al. (2002) and IPCC (2001). The NFA 2011 Edition
switched to time-varying ocean uptake from Khatiwala et al. (Borucke et al. 2013, §3.5).

### AFCS: A Key Parameter Update

Lin et al. (2018) report that updating the AFCS value had the **largest single impact** on NFA
results, changing the average country Footprint by +13.1% (up to +28.9% for some countries).
The 0.73 ± 0.37 t C/ha/yr value from Mancini et al. (2016) distinguishes three forest types
(primary, secondary, plantations) and represents a more conservative sequestration estimate than
earlier NFA editions.

### Calculation Steps

1. Get CO₂ emissions from `co2_energy_faostat.csv` for USA 2023
2. Convert kilotonnes → tonnes
3. Multiply by (1 − 0.35) to get land-fraction CO₂
4. Divide by sequestration rate (AFCS × 44/12 = 2.677 t CO₂/ha/yr) to get forest area
5. Multiply by EQF (1.3) → EF in gha

In [30]:
# Step 1: USA 2023 CO2 emissions from energy
usa_co2 = co2_energy[
    (co2_energy["Area Code"] == DEMO_COUNTRY) & (co2_energy["Year"] == DEMO_YEAR)
]
print("USA 2023 CO2 emissions from energy:")
display(usa_co2[["Item", "Element", "Value", "Unit"]])

co2_kt = usa_co2["Value"].sum()
print(f"\nTotal CO2: {co2_kt:,.2f} kt")

USA 2023 CO2 emissions from energy:


,Item,Element,Value,Unit
11612,Energy,Emissions (CO2),"4,640,000.0000",kt



Total CO2: 4,640,000.00 kt


In [31]:
# Step 2: Convert through the formula step-by-step
co2_tonnes = co2_kt * 1000
print(f"Step 1: kt to tonnes:     {co2_kt:,.2f} x 1000 = {co2_tonnes:,.0f} t CO2")

land_co2 = co2_tonnes * (1 - OCEAN_CO2_FRAC)
print(f"Step 2: Land fraction:    {co2_tonnes:,.0f} x (1 - {OCEAN_CO2_FRAC}) = {land_co2:,.0f} t CO2")

seq_rate_co2 = AFCS * (44.0 / 12.0)
print(f"Step 3: Seq rate (CO2):   {AFCS} x (44/12) = {seq_rate_co2:,.4f} t CO2/ha/yr")

carbon_area_ha = land_co2 / seq_rate_co2
print(f"Step 4: Area needed:      {land_co2:,.0f} / {seq_rate_co2:,.4f} = {carbon_area_ha:,.0f} ha")

ef_carbon = carbon_area_ha * EQF_CARBON
print(f"Step 5: Apply EQF:        {carbon_area_ha:,.0f} x {EQF_CARBON} = {ef_carbon:,.0f} gha")

stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_carbon_gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_carbon:,.0f} gha")
print(f"Match: {abs(ef_carbon - stored_val) / stored_val < 0.001}")

Step 1: kt to tonnes:     4,640,000.00 x 1000 = 4,640,000,000 t CO2
Step 2: Land fraction:    4,640,000,000 x (1 - 0.35) = 3,016,000,000 t CO2
Step 3: Seq rate (CO2):   0.73 x (44/12) = 2.6767 t CO2/ha/yr
Step 4: Area needed:      3,016,000,000 / 2.6767 = 1,126,774,595 ha
Step 5: Apply EQF:        1,126,774,595 x 1.3 = 1,464,806,974 gha

Stored result:  1,464,806,974 gha
Computed here:  1,464,806,974 gha
Match: True


---
## Section 13: Assembly & Final Results

All 6 EF and 5 BC components assembled into totals.

### Aggregation Formulas

$$\text{EF}_{\text{total}} = \text{EF}_{\text{cropland}} + \text{EF}_{\text{grazing}} + \text{EF}_{\text{forest}} + \text{EF}_{\text{fishing}} + \text{EF}_{\text{built-up}} + \text{EF}_{\text{carbon}}$$

$$\text{BC}_{\text{total}} = \text{BC}_{\text{cropland}} + \text{BC}_{\text{grazing}} + \text{BC}_{\text{forest}} + \text{BC}_{\text{fishing}} + \text{BC}_{\text{built-up}}$$

$$\text{Ecological Deficit} = \text{BC}_{\text{total}} - \text{EF}_{\text{total}}$$

A **negative** deficit means ecological overshoot: the country consumes more biocapacity than it has.
A **positive** value means ecological reserve.

### Per-Capita Metrics

$$\text{EF per capita} = \frac{\text{EF}_{\text{total}}}{\text{Population}}$$

Population is from FAOSTAT (in thousands, multiplied by 1000 to get actual count). Per-capita metrics
allow fair comparison between countries of different sizes.

### Global Metrics

> *"Humanity's Ecological Footprint first exceeded global biocapacity in the 1970s, and this
> 'ecological overshoot' has been increasing ever since."*
> — Wackernagel et al. (2002)

$$\text{Number of Earths} = \frac{\sum_{\text{all countries}} \text{EF}_{\text{total}}}{\sum_{\text{all countries}} \text{BC}_{\text{total}}}$$

$$\text{Overshoot Day} = \lfloor 365 \times \frac{\text{World BC}}{\text{World EF}} \rfloor$$

Overshoot Day is the calendar date by which humanity has used nature's entire annual budget. Earlier
dates indicate greater overshoot. A value of 365 would mean humanity lives within Earth's means.

### Conservative Bias

> *"NFAs are specifically constructed to yield conservative estimates of global overshoot. On the
> supply side, biocapacity is overestimated... On the demand side, Ecological Footprint is
> underestimated as it does not track freshwater consumption, soil erosion, or GHG emissions
> other than CO₂."*
> — Borucke et al. (2013)

The true ecological overshoot is likely *larger* than what the accounts show, because many forms
of environmental degradation (water depletion, soil loss, biodiversity loss, non-CO₂ greenhouse
gases) are not yet captured in the methodology.

### What this section shows

1. All 6 EF + 5 BC components for USA 2023 in one summary table
2. Per-capita values compared to stored results
3. Detailed validation: computed vs. stored for every component
4. Multi-country comparison (USA, China, India, Brazil)
5. Global summary: world EF, world BC, Number of Earths, Overshoot Day

In [32]:
# Assemble all components for USA 2023
component_data = pd.DataFrame({
    "Component": ["Cropland", "Grazing", "Forest", "Fishing", "Built-up", "Carbon"],
    "EF (gha)": [ef_cropland, ef_grazing, ef_forest, ef_fishing, ef_built_up, ef_carbon],
    "BC (gha)": [bc_cropland, bc_grazing, bc_forest, bc_fishing, bc_built_up, 0],
})
component_data["EF (gha)"] = component_data["EF (gha)"].round(0)
component_data["BC (gha)"] = component_data["BC (gha)"].round(0)

ef_total = component_data["EF (gha)"].sum()
bc_total = component_data["BC (gha)"].sum()
deficit = bc_total - ef_total

# Add totals row
totals = pd.DataFrame({
    "Component": ["TOTAL"],
    "EF (gha)": [ef_total],
    "BC (gha)": [bc_total],
})
summary = pd.concat([component_data, totals], ignore_index=True)

print("USA 2023 \u2014 All Components:")
display(summary.style.format({"EF (gha)": "{:,.0f}", "BC (gha)": "{:,.0f}"}))

print(f"\nEcological Deficit: {deficit:,.0f} gha")

USA 2023 — All Components:


,Component,EF (gha),BC (gha)
0,Cropland,"296,960,827","350,150,978"
1,Grazing,"43,292,202","133,403,121"
2,Forest,"279,411,154","401,794,614"
3,Fishing,"122,282,182","14,156,665"
4,Built-up,"29,109,650","29,109,650"
5,Carbon,"1,464,806,974",0
6,TOTAL,"2,235,862,989","928,615,028"



Ecological Deficit: -1,307,247,961 gha


In [33]:
# Per capita
usa_pop_row = population[
    (population["Area Code"] == DEMO_COUNTRY)
    & (population["Year"] == DEMO_YEAR)
    & (population["Element"] == "Total Population - Both sexes")
]
pop_thousands = usa_pop_row["Value"].iloc[0]
pop_actual = pop_thousands * 1000

ef_per_cap = ef_total / pop_actual
bc_per_cap = bc_total / pop_actual

stored_ef_total = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_total_gha")
stored_bc_total = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_total_gha")
stored_ef_pc = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_per_capita_gha")
stored_bc_pc = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_per_capita_gha")

print(f"USA population: {pop_thousands:,.1f} thousand = {pop_actual:,.0f} people")
print(f"\nEF total:  computed {ef_total:,.0f}  |  stored {stored_ef_total:,.0f}")
print(f"BC total:  computed {bc_total:,.0f}  |  stored {stored_bc_total:,.0f}")
print(f"EF/cap:    computed {ef_per_cap:,.4f}  |  stored {stored_ef_pc:,.4f}")
print(f"BC/cap:    computed {bc_per_cap:,.4f}  |  stored {stored_bc_pc:,.4f}")

USA population: 343,477.3 thousand = 343,477,335 people

EF total:  computed 2,235,862,989  |  stored 2,235,862,989
BC total:  computed 928,615,028  |  stored 928,615,028
EF/cap:    computed 6.5095  |  stored 6.5095
BC/cap:    computed 2.7036  |  stored 2.7036


In [34]:
# Compare with stored values for validation
print("Detailed comparison with stored results:")
cols = [
    "ef_cropland_gha", "ef_grazing_gha", "ef_forest_gha",
    "ef_fishing_gha", "ef_built_up_gha", "ef_carbon_gha",
    "ef_total_gha",
    "bc_cropland_gha", "bc_grazing_gha", "bc_forest_gha",
    "bc_fishing_gha", "bc_built_up_gha", "bc_total_gha",
]
computed = [
    ef_cropland, ef_grazing, ef_forest, ef_fishing, ef_built_up, ef_carbon,
    ef_total,
    bc_cropland, bc_grazing, bc_forest, bc_fishing, bc_built_up, bc_total,
]
stored_vals = [stored(DEMO_COUNTRY, DEMO_YEAR, c) for c in cols]

comparison = pd.DataFrame({
    "Component": cols,
    "Computed": computed,
    "Stored": stored_vals,
})
comparison["Diff (%)"] = (
    (comparison["Computed"] - comparison["Stored"]) / comparison["Stored"].replace(0, np.nan) * 100
)
display(comparison.style.format({"Computed": "{:,.0f}", "Stored": "{:,.0f}", "Diff (%)": "{:+.3f}%"}))

Detailed comparison with stored results:


,Component,Computed,Stored,Diff (%)
0,ef_cropland_gha,"296,960,827","296,960,827",+0.000%
1,ef_grazing_gha,"43,292,202","43,292,202",+0.000%
2,ef_forest_gha,"279,411,154","279,411,154",+0.000%
3,ef_fishing_gha,"122,282,182","122,282,182",+0.000%
4,ef_built_up_gha,"29,109,650","29,109,650",+0.000%
5,ef_carbon_gha,"1,464,806,974","1,464,806,974",+0.000%
6,ef_total_gha,"2,235,862,989","2,235,862,989",-0.000%
7,bc_cropland_gha,"350,150,978","350,150,978",+0.000%
8,bc_grazing_gha,"133,403,121","133,403,121",-0.000%
9,bc_forest_gha,"401,794,614","401,794,614",+0.000%


In [35]:
# Summary for additional countries: China, India, Brazil
demo_countries = {
    231: "USA",
    41: "China, mainland",
    100: "India",
    21: "Brazil",
}

multi = results[
    (results["area_code"].isin(demo_countries.keys()))
    & (results["year"] == DEMO_YEAR)
][
    ["area_code", "area", "year",
     "ef_cropland_gha", "ef_grazing_gha", "ef_forest_gha",
     "ef_fishing_gha", "ef_built_up_gha", "ef_carbon_gha",
     "ef_total_gha", "bc_total_gha", "ecological_deficit_gha",
     "population", "ef_per_capita_gha", "bc_per_capita_gha"]
].copy()

print(f"Comparison: Multiple countries ({DEMO_YEAR})")
display(
    multi.style.format({
        col: "{:,.0f}" for col in multi.columns
        if col not in ["area_code", "area", "year", "ef_per_capita_gha", "bc_per_capita_gha"]
    }).format({"ef_per_capita_gha": "{:.2f}", "bc_per_capita_gha": "{:.2f}"})
)

Comparison: Multiple countries (2023)


,area_code,area,year,ef_cropland_gha,ef_grazing_gha,ef_forest_gha,ef_fishing_gha,ef_built_up_gha,ef_carbon_gha,ef_total_gha,bc_total_gha,ecological_deficit_gha,population,ef_per_capita_gha,bc_per_capita_gha
199,21,Brazil,2023,174422947.579298,118346297.500612,226994364.640884,13892015.942657,4184268.000000,132590286.425903,670430180.089355,813969353.627100,143539173.537745,211140.729000,3.18,3.86
359,41,"China, mainland",2023,485756678.194491,79531151.279592,233546754.640884,495907354.546464,8420280.000000,3504171855.541719,4807334074.203151,1202576873.792997,-3604757200.410154,1422584.933000,3.38,0.85
849,100,India,2023,310848913.982173,174041046.035714,249765784.475138,81444127.005369,11109318.000000,918661270.236613,1745870459.735007,399284548.026475,-1346585911.708532,1438069.596000,1.21,0.28
2049,231,United States of America,2023,296960827.319610,43292201.632653,279411153.977901,122282182.254305,29109650.240000,1464806973.848070,2235862989.272539,928615027.825781,-1307247961.446758,343477.335000,6.51,2.70


In [36]:
# Global summary: sum all countries -> Number of Earths, Overshoot Day
gs_2023 = global_summary[global_summary["year"] == DEMO_YEAR].iloc[0]

# Recompute from country data
all_2023 = results[results["year"] == DEMO_YEAR]
world_ef = all_2023["ef_total_gha"].sum()
world_bc = all_2023["bc_total_gha"].sum()
n_earths = world_ef / world_bc
overshoot_day = math.floor(365 * world_bc / world_ef)

from datetime import datetime, timedelta
overshoot_date = datetime(DEMO_YEAR, 1, 1) + timedelta(days=overshoot_day - 1)

print("=" * 70)
print(f"GLOBAL SUMMARY ({DEMO_YEAR})")
print("=" * 70)
print(f"World EF:          {world_ef / 1e9:>10.2f} billion gha")
print(f"World BC:          {world_bc / 1e9:>10.2f} billion gha")
print(f"Overshoot:         {(world_ef - world_bc) / 1e9:>10.2f} billion gha")
print(f"Number of Earths:  {n_earths:>10.2f}")
print(f"Overshoot Day:     {overshoot_date.strftime('%B %d')} (day {overshoot_day})")
print(f"\nStored values:     {gs_2023['number_of_earths']:.2f} Earths, day {int(gs_2023['overshoot_day'])}")
print(f"Match: {int(gs_2023['overshoot_day']) == overshoot_day}")

GLOBAL SUMMARY (2023)
World EF:               24.56 billion gha
World BC:               11.84 billion gha
Overshoot:              12.71 billion gha
Number of Earths:        2.07
Overshoot Day:     June 25 (day 176)

Stored values:     2.07 Earths, day 176
Match: True
